# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(metadata.name + ':')
print(metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# We'll list available record sets, the fields within them, and example record schema.

from pprint import pprint

print("Available record sets (using their '@id'):")
for record_set in metadata.record_sets:
    print(f"  - {record_set['@id']} (name: {record_set.get('name', '')})")

print("\nFields and columns for each record set:")
for record_set in metadata.record_sets:
    print(f"\nRecord set '@id': {record_set['@id']}")
    fields = record_set.get('fields', [])
    for field in fields:
        print(f"    Field '@id': {field['@id']} | name: {field.get('name', '')} | dataType: {field.get('dataType', '')}")
        if 'columns' in field:
            for column in field['columns']:
                print(f"      Column '@id': {column['@id']} | name: {column.get('name', '')}")

# Display the first few records from each record set
for record_set in metadata.record_sets:
    print(f"\nSample data for record set: {record_set['@id']}")
    try:
        for i, rec in enumerate(dataset.records(record_set=record_set['@id'])):
            pprint(rec)
            if i >= 2:
                break
    except Exception as e:
        print(f"Could not load records for {record_set['@id']}: {e}")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Define the record sets to extract. Replace these example '@id's after running the overview if necessary.
record_sets_ids = [
    rs['@id'] for rs in metadata.record_sets
]

dataframes = {}

for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id} with columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load DataFrame for {record_set_id}: {e}")

# Display the columns and preview the first record set
if record_sets_ids:
    first_id = record_sets_ids[0]
    print(f"\nColumns in first record set '{first_id}':")
    print(dataframes[first_id].columns.tolist())
    print(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
You must use the `@id`s for fields. If you are not sure which field to use, refer to the overview step.

In [ ]:
# Example: Filter and normalize based on a numeric field in the main protein abundance record set

main_record_set_id = record_sets_ids[0] # Update if needed
df = dataframes[main_record_set_id]

# Select a numeric field for analysis
# Replace with a valid field '@id' from the overview (example shown)
numeric_field_id = None
for col in df.columns:
    # Choose the first float/integer column by column name
    if df[col].dtype in [int, float, 'int64', 'float64']:
        numeric_field_id = col
        break

if numeric_field_id is None:
    # Try to convert a likely numeric column
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            numeric_field_id = col
            break
        except Exception:
            continue

if numeric_field_id is not None:
    print(f'Using numeric field: {numeric_field_id}')

    threshold = df[numeric_field_id].mean() if numeric_field_id in df else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a likely categorical field (e.g., protein description)
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (means):")
        print(grouped_df.head())
else:
    print("No numeric field identified for EDA. Please review available columns in previous cell.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
if numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field if available
    if group_field is not None and group_field in df.columns:
        top_n_values = df[group_field].value_counts().index[:10]
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df[df[group_field].isin(top_n_values)], x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field} (Top 10 {group_field}s)")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field detected for visualization.")

## 6. Conclusion
We successfully loaded the FAIR^2 dataset describing mass spectrometry analysis of extracellular vesicles isolated from stimulated human mast cells using the `mlcroissant` library. By exploring the dataset via record set and field `@id`s, we were able to extract key numeric fields, filter and visualize data distributions, and group by significant attributes. 

**To continue, customize field and record set `@id` selections based on your specific analysis needs. See the overview cells for all available keys.**